# Process the .xls file for cities with >100k population

Process the file table08.xls from United Nation's Department of Economic and Social Affairs [Statistics Division](https://unstats.un.org/unsd/demographic-social/products/dyb/dyb_2022/).

# Setup

In [129]:
%run -i "functions.py"

## Process the table

In [109]:
un_list = pd.read_excel(
    "../external_data/table08.xls",
    sheet_name = "Data",
    skiprows = 6,
    usecols = "A:B,J",
    engine = "xlrd"
).rename(columns = {"Unnamed: 0": "name", "Unnamed: 1": "pop_city", "Unnamed: 9": "pop_urban_area"})
un_list

,name,pop_city,pop_urban_area
0,AFRICA - AFRIQUE,NaN,NaN
1,Algeria - Algérie,NaN,NaN
2,16 IV 2008 (CDJC),NaN,NaN
3,Adrar,200834,...
4,Ain Defla,450280,...
...,...,...,...
4439,16 XI 2020 (CDFC),NaN,NaN
4440,PORT VILA,49034,...
4441,Wallis and Futuna Islands - Îles Wallis et Futuna,NaN,NaN
4442,21 VII 2008 (CDFC),NaN,NaN


In [ ]:
records = []

current_continent = ""
current_country = ""
city = ""

info_row = re.compile(r"^\d") # Rows stating the source and year start with a digit

cities_with_no_population_data = [] # List to keep track of cities with no population data


for idx, row in tqdm(un_list.iterrows(), total=len(un_list), position=0, leave=True):

    if pd.isna(row["name"]):
        continue


    caps = row["name"].isupper()
    

    # Check if it is a continent (all caps with no population data and the first element is not a digit):
    if caps and pd.isna(row["pop_city"]) and info_row.match(row["name"]) is None:
        continent = re.sub(r"[^a-zA-Z ]", "", row["name"].split(" - ")[0])
        current_continent = continent.title()
        print(f"Processing continent: {continent}")
        continue


    # Check if it is a country row (no population data and the first element is not a digit):
    if info_row.match(row["name"]) is None and pd.isna(row["pop_city"]):
        country = footnote.sub("", row["name"].split(" - ")[0]).strip()
        current_country = country.title()
        continue


    # Check if it is an information row (the first element is a digit):
    if info_row.match(row["name"]) is not None:
        continue

    capital = True if caps else False

    city = re.sub(r"\d+$", "", re.sub(r"\(.*?\)", "", row["name"])).strip().title() # Clean the city name: remove footnotes, parenthetical information, and write with uppercase first letter of each word

    if pd.isna(row["pop_city"]) or not isinstance(row["pop_city"], (int, float)):
        if pd.isna(row["pop_urban_area"]) or not isinstance(row["pop_urban_area"], (int, float)):
            cities_with_no_population_data.append(city)
            continue
        else:
            pop = int(row["pop_urban_area"])
    else:
        pop = int(row["pop_city"])
    

    records.append({
        "continent": current_continent,
        "country": current_country,
        "city": city,
        "is_capital": capital,
        "population": pop
    })

100%|██████████| 4444/4444 [00:00<00:00, 28478.80it/s]

Processing continent: AFRICA
Processing continent: AMERICA NORTH
Processing continent: AMERICA SOUTH
Processing continent: ASIA
Processing continent: EUROPE
Processing continent: OCEANIA


In [111]:
un_list_clean = pd.DataFrame(
    records,
    columns = ["continent", "country", "city", "is_capital", "population"]
)
un_list_clean

,continent,country,city,is_capital,population
0,Africa,Algeria,Adrar,False,200834
1,Africa,Algeria,Ain Defla,False,450280
2,Africa,Algeria,Ain Temouchent,False,299341
3,Africa,Algeria,Algiers,True,2712944
4,Africa,Algeria,Annaba,False,442230
...,...,...,...,...,...
4020,Oceania,Solomon Islands,Honiara,True,64609
4021,Oceania,Tonga,Nuku'Alofa,True,34142
4022,Oceania,Tuvalu,Funafuti,True,6320
4023,Oceania,Vanuatu,Port Vila,True,49034


In [112]:
un_list_clean['city'] = (
    un_list_clean['city']
    .str.replace("ş", "ș", regex=False)
    .str.replace("Ş", "Ș", regex=False)
    .str.replace("ţ", "ț", regex=False)
    .str.replace("Ţ", "Ț", regex=False)
)

In [113]:
# Save the list
un_list_clean.to_csv("../external_data/table08_clean.csv", index=False)

## EUROPE

Our definition of Europe includes (39):
* Albania
* Andorra
* Austria
* Belarus
* Belgium
* Bosnia and Herzegovina
* Bulgaria
* Croatia
* Czech Republic
* Denmark
* Estonia
* Finland
* France
* Germany
* Greece
* Hungary
* Iceland
* Ireland
* Italy
* Kosovo
* Latvia
* Lithuania
* Luxembourg
* Moldova
* Montenegro
* Netherlands
* North Macedonia
* Norway
* Poland
* Portugal
* Romania
* Serbia
* Slovakia
* Slovenia
* Spain
* Sweden
* Switzerland
* Ukraine
* United Kingdom

and excludes (11):
* Armenia
* Azerbaijan
* Cyprus
* Georgia
* Liechtenstein
* Malta
* Monaco
* Russia
* San Marino
* Turkiye
* Vatican City

### Import data

In [114]:
# Import the list of European capitals already extracted
european_capitals = pd.read_csv("../cities/european_capitals.csv", sep = ";")
capitals = european_capitals['name_en'].tolist()
countries = sorted(european_capitals['country_en'].tolist())

In [115]:
# Import the cleaned UN list and filter for European countries
europe = pd.read_csv("../external_data/table08_clean.csv", sep = ",").query("continent == 'Europe'").drop(columns="continent").reset_index(drop=True)
europe

,country,city,is_capital,population
0,Åland Islands,Mariehamn,True,11750
1,Albania,Durrës,False,113249
2,Albania,Tirana,True,418495
3,Andorra,Andorra La Vella,True,22205
4,Austria,Graz,False,288806
...,...,...,...,...
691,United Kingdom Of Great Britain And Northern I...,Reading,False,218705
692,United Kingdom Of Great Britain And Northern I...,Sheffield,False,518090
693,United Kingdom Of Great Britain And Northern I...,Southampton,False,253651
694,United Kingdom Of Great Britain And Northern I...,Stoke-On-Trent,False,270726


In [116]:
europe['city'].to_csv("../external_data/table08_europe_cities.csv", index=False, header=False)

We used Claude to check each individal city name for the English spelling.

In [117]:
cities_eng = pd.read_csv("../external_data/table08_europe_cities_english.csv")
europe = europe.merge(
    cities_eng,
    on="city",
    how="left"
)
europe

,country,city,is_capital,population,city_eng
0,Åland Islands,Mariehamn,True,11750,Mariehamn
1,Albania,Durrës,False,113249,Durrës
2,Albania,Tirana,True,418495,Tirana
3,Andorra,Andorra La Vella,True,22205,Andorra La Vella
4,Austria,Graz,False,288806,Graz
...,...,...,...,...,...
693,United Kingdom Of Great Britain And Northern I...,Reading,False,218705,Reading
694,United Kingdom Of Great Britain And Northern I...,Sheffield,False,518090,Sheffield
695,United Kingdom Of Great Britain And Northern I...,Southampton,False,253651,Southampton
696,United Kingdom Of Great Britain And Northern I...,Stoke-On-Trent,False,270726,Stoke-On-Trent


### Match the different sources

In [118]:
# Rename the countries to match the european_capital.csv
europe['country'] = europe['country'].replace({
    'United Kingdom Of Great Britain And Northern Ireland': 'United Kingdom',
    'Netherlands (Kingdom Of The)': 'Netherlands',
    "Republic Of Moldova": "Moldova",
    "Czechia": "Czech Republic"
})

In [119]:
# Checks
print("Countries in the UN list:")
print(set(europe['country'].unique()))

print()
print("Countries in the European capitals list that are not in the UN list:")
print(set(countries) - set(europe['country'].unique()))

print()
print("Countries in the UN list that are not in the European capitals list:")
print(set(europe['country'].unique()) - set(countries))

Countries in the UN list:
{'Belarus', 'Slovakia', 'Iceland', 'Denmark', 'Croatia', 'Guernsey', 'Latvia', 'Malta', 'Belgium', 'United Kingdom', 'San Marino', 'Liechtenstein', 'Czech Republic', 'Monaco', 'Isle Of Man', 'Jersey', 'Holy See', 'Faroe Islands', 'North Macedonia', 'France', 'Åland Islands', 'Estonia', 'Greece', 'Poland', 'Italy', 'Netherlands', 'Sweden', 'Ireland', 'Moldova', 'Austria', 'Romania', 'Hungary', 'Andorra', 'Finland', 'Ukraine', 'Portugal', 'Russian Federation', 'Lithuania', 'Germany', 'Bulgaria', 'Luxembourg', 'Norway', 'Spain', 'Albania', 'Slovenia', 'Switzerland', 'Serbia', 'Gibraltar', 'Montenegro'}

Countries in the European capitals list that are not in the UN list:
{'Bosnia and Herzegovina', 'Kosovo'}

Countries in the UN list that are not in the European capitals list:
{'Monaco', 'Russian Federation', 'Isle Of Man', 'Jersey', 'Holy See', 'Faroe Islands', 'Guernsey', 'Åland Islands', 'Malta', 'San Marino', 'Liechtenstein', 'Gibraltar'}


In [120]:
# Filter the UN list for the countries in the European capitals list
europe_filtered = europe.query("country in @countries").reset_index(drop=True)
europe_filtered

,country,city,is_capital,population,city_eng
0,Albania,Durrës,False,113249,Durrës
1,Albania,Tirana,True,418495,Tirana
2,Andorra,Andorra La Vella,True,22205,Andorra La Vella
3,Austria,Graz,False,288806,Graz
4,Austria,Innsbruck,False,132110,Innsbruck
...,...,...,...,...,...
507,United Kingdom,Reading,False,218705,Reading
508,United Kingdom,Sheffield,False,518090,Sheffield
509,United Kingdom,Southampton,False,253651,Southampton
510,United Kingdom,Stoke-On-Trent,False,270726,Stoke-On-Trent


In [121]:
# Checks
print("Countries in the European capitals list that are not in the filtered:")
print(set(countries) - set(europe_filtered['country'].unique()))

print()
print("Countries in the filtered list that are not in the European capitals list:")
print(set(europe_filtered['country'].unique()) - set(countries))

Countries in the European capitals list that are not in the filtered:
{'Bosnia and Herzegovina', 'Kosovo'}

Countries in the filtered list that are not in the European capitals list:
set()


In [122]:
# Rename the capitals to match the european_capital.csv
europe_filtered['city_eng'] = europe_filtered['city_eng'].replace({
    "Andorra La Vella": "Andorra la Vella",
    "Reykjavik": "Reykjavík",
    "Luxembourg City": "Luxembourg",
    'Chisinau': "Chișinău"
})

In [123]:
europe_capitals = europe_filtered.query("is_capital == True").reset_index(drop=True)
print("Capitals in the European capitals list that are not in the filtered:")
print(set(capitals) - set(europe_capitals['city_eng'].unique()))

print()
print("Capitals in the filtered list that are not in the European capitals list:")
print(set(europe_capitals['city_eng'].unique()) - set(capitals))

Capitals in the European capitals list that are not in the filtered:
{'Pristina', 'Sarajevo'}

Capitals in the filtered list that are not in the European capitals list:
set()


In [124]:
# Remove capitals
europe_non_capitals = europe_filtered.query("is_capital == False").reset_index(drop=True)
europe_non_capitals

,country,city,is_capital,population,city_eng
0,Albania,Durrës,False,113249,Durrës
1,Austria,Graz,False,288806,Graz
2,Austria,Innsbruck,False,132110,Innsbruck
3,Austria,Klagenfurt,False,100817,Klagenfurt
4,Austria,Linz,False,205726,Linz
...,...,...,...,...,...
470,United Kingdom,Reading,False,218705,Reading
471,United Kingdom,Sheffield,False,518090,Sheffield
472,United Kingdom,Southampton,False,253651,Southampton
473,United Kingdom,Stoke-On-Trent,False,270726,Stoke-On-Trent


### Manual add the missing countries

There a few european countries missing from the UN list:
- Bosnia and Herzegovina
- Kosovo

We add the cities with more than 100k citizens from [Wikipedia](https://en.wikipedia.org/wiki/List_of_towns_and_cities_with_100,000_or_more_inhabitants/country:_A-B).

In this case, there is only one, since the other cities from these two countries with more than 100k citizens are the capitals. 

In [125]:
europe_non_capitals.loc[-1] = ["Bosnia And Herzegovina", "Banja Luka", True, 135059, "Banja Luka"]
europe_non_capitals.index = europe_non_capitals.index + 1  # shifting index
europe_non_capitals = europe_non_capitals.sort_values("country").reset_index(drop=True)
europe_non_capitals

,country,city,is_capital,population,city_eng
0,Albania,Durrës,False,113249,Durrës
1,Austria,Graz,False,288806,Graz
2,Austria,Innsbruck,False,132110,Innsbruck
3,Austria,Klagenfurt,False,100817,Klagenfurt
4,Austria,Linz,False,205726,Linz
...,...,...,...,...,...
471,United Kingdom,Belfast,False,280211,Belfast
472,United Kingdom,Aberdeen,False,207932,Aberdeen
473,United Kingdom,Wolverhampton,False,210319,Wolverhampton
474,United Kingdom,Leeds,False,474632,Leeds


### Extract final csv

In [130]:
european_100000pop = europe_non_capitals.drop(
        columns = ["is_capital", "city"]
    ).rename(
        columns = {
            "country": "country_en",
            "city_eng": "name_en"
        }
    )

european_100000pop['nominatim_query'] = european_100000pop['name_en']
european_100000pop["alpha-2"] = european_100000pop["country_en"].map(get_alpha2)
european_100000pop

,country_en,population,name_en,nominatim_query,alpha-2
0,Albania,113249,Durrës,Durrës,AL
1,Austria,288806,Graz,Graz,AT
2,Austria,132110,Innsbruck,Innsbruck,AT
3,Austria,100817,Klagenfurt,Klagenfurt,AT
4,Austria,205726,Linz,Linz,AT
...,...,...,...,...,...
471,United Kingdom,280211,Belfast,Belfast,GB
472,United Kingdom,207932,Aberdeen,Aberdeen,GB
473,United Kingdom,210319,Wolverhampton,Wolverhampton,GB
474,United Kingdom,474632,Leeds,Leeds,GB


In [133]:
european_100000pop = european_100000pop[["name_en", "country_en", "alpha-2", "population", "nominatim_query"]]
european_100000pop.to_csv("../cities/european_100000pop.csv", index=False)